In [105]:
from sqlalchemy import Engine, MetaData, Table, cast, func, update
from sqlalchemy.sql.sqltypes import BigInteger, DOUBLE_PRECISION, Integer, SmallInteger, Numeric, Double

import os
from pathlib import Path
import pandas as pd


from sql_utils import drop_table, build_postgres_engine


project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists() and project_root != project_root.parent:
    project_root = project_root.parent

engine = build_postgres_engine(
        "localhost",
        int(os.environ.get("POSTGRES_PORT", 5432)),
        os.environ["POSTGRES_DB"],
        os.environ["POSTGRES_USER"],
        os.environ["POSTGRES_PASSWORD"],
    )
    
chunck_size = 100_000 # Se estiver estourando mt ram, abaixe esse numero

In [106]:
from sqlalchemy import Column, MetaData, Table, insert, inspect, null, select, String, delete, text
from sqlalchemy.schema import CreateSchema
from sqlalchemy.sql.type_api import TypeEngine

from sql_utils import normalize_table_columns_by_factor

source_schema = "raw"
target_schema_name = "sinan_viol"

def copy_table_between_schemas(
    engine: Engine,
    source_schema_name: str,
    source_table_name: str,
    target_schema_name: str,
    target_table_name: str | None = None,
    column_name_mapping: dict[str, str] | None = None,
    extra_columns: dict[str, TypeEngine] | None = None,
) -> None:
    target_table_name = target_table_name or source_table_name
    column_name_mapping = column_name_mapping or {}
    extra_columns = extra_columns or {}

    inspector = inspect(engine)
    if inspector.has_table(target_table_name, schema=target_schema_name):
        raise ValueError(f"Target table already exists: {target_schema_name}.{target_table_name}")

    source_metadata = MetaData(schema=source_schema_name)
    source_table = Table(source_table_name, source_metadata, autoload_with=engine)

    target_metadata = MetaData(schema=target_schema_name)
    target_columns = [
        Column(
            column_name_mapping.get(column.name, column.name),
            column.type,
            primary_key=column.primary_key,
            nullable=column.nullable,
        )
        for column in source_table.columns
    ]
    target_columns.extend(
        Column(column_name, column_type, nullable=True)
        for column_name, column_type in extra_columns.items()
    )
    target_table = Table(target_table_name, target_metadata, *target_columns)

    source_columns = list(source_table.columns)
    selected_columns = [*source_columns, *[null() for _ in extra_columns]]
    target_column_names = [column.name for column in target_table.columns]

    copy_statement = insert(target_table).from_select(
        target_column_names,
        select(*selected_columns),
    )

    with engine.begin() as connection:
        connection.execute(CreateSchema(target_schema_name, if_not_exists=True))
        target_table.create(connection)
        connection.execute(copy_statement)


def delete_missing_municipality_rows(
    engine: Engine,
    schema_name: str,
    table_name: str,
    territory_code_column: str,
    reference_table_name: str,
    reference_code_column: str,
) -> None:
    metadata = MetaData(schema=schema_name)
    table = Table(table_name, metadata, autoload_with=engine)
    reference_table = Table(reference_table_name, metadata, autoload_with=engine)

    territory_code = cast(table.c[territory_code_column], String)
    reference_code = cast(reference_table.c[reference_code_column], String)

    statement = delete(table).where(
        func.length(territory_code) > 2,
        territory_code.not_in(select(reference_code)),
    )

    with engine.begin() as connection:
        result = connection.execute(statement)

    print(f"Deleted rows: {result.rowcount}")

def delete_rows_below_threshold(
    engine: Engine,
    schema_name: str,
    table_name: str,
    column_name: str,
    threshold: int | float,
) -> None:
    metadata = MetaData(schema=schema_name)
    table = Table(table_name, metadata, autoload_with=engine)

    statement = delete(table).where(table.c[column_name] < threshold)

    with engine.begin() as connection:
        result = connection.execute(statement)

    print(f"Deleted rows: {result.rowcount}")

def rename_table_column(
    engine: Engine,
    schema_name: str,
    table_name: str,
    old_column_name: str,
    new_column_name: str,
) -> None:
    preparer = engine.dialect.identifier_preparer
    qualified_table_name = f"{preparer.quote_schema(schema_name)}.{preparer.quote(table_name)}"
    old_column = preparer.quote(old_column_name)
    new_column = preparer.quote(new_column_name)

    statement = text(
        f"ALTER TABLE {qualified_table_name} RENAME COLUMN {old_column} TO {new_column}"
    )

    with engine.begin() as connection:
        connection.execute(statement)

    print(f"Renamed column: {schema_name}.{table_name}.{old_column_name} -> {new_column_name}")

    


# Apaga o schema para começar limpo

In [107]:
from sqlalchemy import text


def drop_schema(
    engine: Engine,
    schema_name: str,
) -> None:
    preparer = engine.dialect.identifier_preparer
    quoted_schema = preparer.quote_schema(schema_name)

    with engine.begin() as connection:
        connection.execute(text(f"DROP SCHEMA IF EXISTS {quoted_schema} CASCADE"))

    inspector = inspect(engine)
    inspector.clear_cache()
    if inspector.has_schema(schema_name):
        raise RuntimeError(f"Schema still exists after DROP SCHEMA CASCADE: {schema_name}")

    print(f"Dropped schema and all objects inside it: {schema_name}")


# Para usar quando quiser recomeçar:
drop_schema(engine, target_schema_name)


Dropped schema and all objects inside it: sinan_viol


## Tirando um copia

In [108]:
target_table_name = "ocorrencias"
copy_table_between_schemas(engine, source_schema, "SINAN_Violencia", target_schema_name, target_table_name)

Ajusta o codigo de municipio para o cod de 7 digitos do ibge

In [109]:
from sqlalchemy import create_engine, text
import re


target_table_name = "ocorrencias"
municipio_column = ["ID_MUNICIP", "ID_MN_OCOR"]


def quote_ident(identifier: str) -> str:
    if not re.match(r"^[a-zA-Z_][a-zA-Z0-9_]*$", identifier):
        raise ValueError(f"Identificador SQL inválido: {identifier}")
    return f'"{identifier}"'

for mun in municipio_column:

    schema_sql = quote_ident(target_schema_name)
    table_sql = quote_ident(target_table_name)
    column_sql = quote_ident(mun)


    with engine.begin() as conn:
        conn.execute(text("""
            CREATE OR REPLACE FUNCTION public.ibge_municipio_7(codigo_6 integer)
            RETURNS integer AS $$
            DECLARE
                c text;
                soma integer := 0;
                digito integer;
                produto integer;
                dv integer;
                i integer;
            BEGIN
                IF codigo_6 IS NULL THEN
                    RETURN NULL;
                END IF;

                c := lpad(codigo_6::text, 6, '0');

                IF c !~ '^[0-9]{6}$' THEN
                    RETURN NULL;
                END IF;

                FOR i IN 1..6 LOOP
                    digito := substring(c from i for 1)::integer;

                    IF i % 2 = 0 THEN
                        produto := digito * 2;

                        IF produto >= 10 THEN
                            produto := produto - 9;
                        END IF;

                        soma := soma + produto;
                    ELSE
                        soma := soma + digito;
                    END IF;
                END LOOP;

                dv := (10 - (soma % 10)) % 10;

                RETURN (c || dv::text)::integer;
            END;
            $$ LANGUAGE plpgsql IMMUTABLE;
        """))

        conn.execute(text(f"""
            UPDATE {schema_sql}.{table_sql}
            SET {column_sql} = public.ibge_municipio_7({column_sql}::integer)
            WHERE {column_sql} IS NOT NULL
            AND length({column_sql}::text) = 6;
        """))

In [110]:
from sql_utils import print_table_head

target_table_name = "ocorrencias"

print_table_head(engine, target_schema_name, target_table_name, rows=1000)

        id     TP_NOT ID_AGRAVO DT_NOTIFIC SEM_NOT  NU_ANO SG_UF_NOT  ID_MUNICIP  ID_UNIDADE    DT_OCOR SEM_PRI  ANO_NASC  NU_IDADE_N CS_SEXO                 CS_GESTANT  CS_RACA    CS_ESCOL_N SG_UF  ID_MN_RESI ID_PAIS NDUPLIC DT_INVEST ID_OCUPA_N               SIT_CONJUG SG_UF_OCOR  ID_MN_OCOR HORA_OCOR                 LOCAL_OCOR             LOCAL_ESPE OUT_VEZES LES_AUTOP VIOL_FISIC VIOL_PSICO VIOL_TORT VIOL_SEXU VIOL_TRAF VIOL_FINAN VIOL_NEGLI VIOL_INFAN VIOL_LEGAL VIOL_OUTR                   VIOL_ESPEC AG_FORCA AG_ENFOR AG_OBJETO AG_CORTE AG_QUENTE AG_ENVEN  AG_FOGO AG_AMEACA AG_OUTROS                       AG_ESPEC    SEX_ASSEDI    SEX_ESTUPR SEX_PUDOR     SEX_PORNO     SEX_EXPLO     SEX_OUTRO              SEX_ESPEC PEN_ORAL PEN_ANAL PEN_VAGINA LESAO_NAT LESAO_ESPE LESAO_CORP   NUM_ENVOLV REL_SEXUAL  REL_PAI  REL_MAE  REL_PAD REL_CONJ REL_EXCON REL_NAMO REL_EXNAM REL_FILHO REL_DESCO REL_IRMAO REL_CONHEC REL_CUIDA REL_PATRAO REL_INST  REL_POL REL_PROPRI REL_OUTROS     REL_ESPEC     A

In [111]:
target_table_name = "ocorrencias"

with engine.connect() as connection:
    inspector = inspect(connection)
    inspector.clear_cache()

    print(f"schema: {target_schema_name}")
    print(f"table: {target_table_name}")

    for column in inspector.get_columns(target_table_name, schema=target_schema_name):
        nullable = "nullable" if column["nullable"] else "not null"
        print(f"  - {column['name']}: {column['type']} ({nullable})")

schema: sinan_viol
table: ocorrencias
  - id: VARCHAR (not null)
  - TP_NOT: TEXT (nullable)
  - ID_AGRAVO: TEXT (nullable)
  - DT_NOTIFIC: TIMESTAMP (nullable)
  - SEM_NOT: TEXT (nullable)
  - NU_ANO: BIGINT (nullable)
  - SG_UF_NOT: TEXT (nullable)
  - ID_MUNICIP: BIGINT (nullable)
  - ID_UNIDADE: BIGINT (nullable)
  - DT_OCOR: TIMESTAMP (nullable)
  - SEM_PRI: TEXT (nullable)
  - ANO_NASC: BIGINT (nullable)
  - NU_IDADE_N: BIGINT (nullable)
  - CS_SEXO: TEXT (nullable)
  - CS_GESTANT: TEXT (nullable)
  - CS_RACA: TEXT (nullable)
  - CS_ESCOL_N: TEXT (nullable)
  - SG_UF: TEXT (nullable)
  - ID_MN_RESI: BIGINT (nullable)
  - ID_PAIS: TEXT (nullable)
  - NDUPLIC: TEXT (nullable)
  - DT_INVEST: TIMESTAMP (nullable)
  - ID_OCUPA_N: TEXT (nullable)
  - SIT_CONJUG: TEXT (nullable)
  - SG_UF_OCOR: TEXT (nullable)
  - ID_MN_OCOR: BIGINT (nullable)
  - HORA_OCOR: TEXT (nullable)
  - LOCAL_OCOR: TEXT (nullable)
  - LOCAL_ESPE: TEXT (nullable)
  - OUT_VEZES: TEXT (nullable)
  - LES_AUTOP: TEXT

# Contagem

In [112]:
import re
import unicodedata
import pandas as pd
from sqlalchemy import text
from sqlalchemy.engine import Engine


def slugify(value: object) -> str:
    """
    Transforma valores como 'Sexo Feminino', 'Sim', 'Não se aplica'
    em nomes seguros para colunas.
    """
    value = str(value).strip().lower()

    value = unicodedata.normalize("NFKD", value)
    value = "".join(char for char in value if not unicodedata.combining(char))

    value = re.sub(r"[^a-z0-9]+", "_", value)
    value = re.sub(r"_+", "_", value)
    value = value.strip("_")

    return value or "vazio"


def sql_literal(value: object) -> str:
    """
    Escapa valores para uso como literal SQL.
    Aqui os valores vêm do próprio banco, mas ainda assim é bom escapar aspas.
    """
    if value is None:
        return "NULL"

    value = str(value).replace("'", "''")
    return f"'{value}'"


def sql_in_list(values: list[object]) -> str:
    """
    Monta lista SQL para IN (...).
    """
    return ", ".join(sql_literal(value) for value in values)


def quote_ident(identifier: str) -> str:
    """
    Escapa identificadores SQL: schema, tabela, colunas e aliases.
    """
    return '"' + identifier.replace('"', '""') + '"'


def group_values_by_slug(values: list[object]) -> dict[str, list[object]]:
    """
    Agrupa valores brutos diferentes que viram a mesma chave normalizada.
    """
    grouped: dict[str, list[object]] = {}

    for value in values:
        key = slugify(value)
        grouped.setdefault(key, []).append(value)

    return grouped


def get_distinct_values(
    engine: Engine,
    schema: str,
    table: str,
    columns: list[str],
    lesion_column: str = "LES_AUTOP",
) -> dict[str, list[str]]:
    """
    Busca os valores distintos de cada coluna categórica no Postgres,
    já removendo linhas em que LES_AUTOP = 'Sim'.
    """

    values_by_column: dict[str, list[str]] = {}

    for column in columns:
        query = text(f"""
            SELECT DISTINCT {quote_ident(column)} AS value
            FROM {quote_ident(schema)}.{quote_ident(table)}
            WHERE {quote_ident(column)} IS NOT NULL
              AND COALESCE(TRIM(LOWER({quote_ident(lesion_column)})), '') <> 'sim'
            ORDER BY value
        """)

        with engine.connect() as conn:
            values = conn.execute(query).scalars().all()

        values_by_column[column] = values

    return values_by_column


def get_analysis_columns() -> tuple[list[str], list[str], list[str]]:
    """
    Retorna:
    - violence_columns
    - sexual_violence_columns
    - categorical_columns

    IMPORTANTE:
    SEX_ESPEC foi removida porque é texto livre e não faz sentido
    gerar contagens dummy diretas para ela.
    """

    violence_columns = [
        "VIOL_FISIC",
        "VIOL_PSICO",
        "VIOL_TORT",
        "VIOL_SEXU",
        "VIOL_TRAF",
        "VIOL_FINAN",
        "VIOL_NEGLI",
        "VIOL_INFAN",
        "VIOL_LEGAL",
        "VIOL_OUTR",
    ]

    sexual_violence_columns = [
        "SEX_ASSEDI",
        "SEX_ESTUPR",
        "SEX_PUDOR",
        "SEX_PORNO",
        "SEX_EXPLO",
        "SEX_OUTRO",
        # "SEX_ESPEC" removida
    ]

    categorical_columns = [
        "CS_SEXO",
        "LOCAL_OCOR",
        *violence_columns,
        *sexual_violence_columns,
        "AUTOR_SEXO",
        "ORIENT_SEX",
        "IDENT_GEN",
        "VIOL_MOTIV",
    ]

    return violence_columns, sexual_violence_columns, categorical_columns


def build_monthly_violence_query(
    schema: str,
    table: str,
    values_by_column: dict[str, list[str]],
    date_column: str = "DT_OCOR",
    municipality_column: str = "ID_MN_OCOR",
    state_column: str = "SG_UF_OCOR",
    lesion_column: str = "LES_AUTOP",
    sex_column: str = "CS_SEXO",
) -> str:
    """
    Monta uma query SQL com:

    - contagem mensal por município e UF;
    - coluna Total;
    - colunas unitárias no formato NOME_COLUNA_CHAVE_NORMALIZADA;
    - colunas combinadas no formato CS_SEXO_CHAVE__OUTRA_COLUNA_CHAVE.

    SEX_ESPEC foi removida.
    """

    violence_columns, sexual_violence_columns, categorical_columns = get_analysis_columns()

    pair_columns = [
        "LOCAL_OCOR",
        "AUTOR_SEXO",
        *violence_columns,
        *sexual_violence_columns,
        "ORIENT_SEX",
        "IDENT_GEN",
    ]

    select_parts = [
        f"DATE_TRUNC('month', {quote_ident(date_column)})::date AS mes",
        quote_ident(municipality_column),
        quote_ident(state_column),
        'COUNT(*) AS "Total"',
    ]

    # ============================================================
    # Contagens unitárias por chave normalizada
    # ============================================================

    for column in categorical_columns:
        grouped_values = group_values_by_slug(
            values_by_column.get(column, [])
        )

        for normalized_key, raw_values in grouped_values.items():
            alias = f"{column}_{normalized_key}"

            select_parts.append(
                f"""COUNT(*) FILTER (
                    WHERE {quote_ident(column)} IN ({sql_in_list(raw_values)})
                ) AS {quote_ident(alias)}"""
            )

    # ============================================================
    # Contagens combinadas:
    # CS_SEXO + LOCAL_OCOR
    # CS_SEXO + AUTOR_SEXO
    # CS_SEXO + VIOL*
    # CS_SEXO + SEX*, exceto SEX_ESPEC
    # CS_SEXO + ORIENT_SEX
    # CS_SEXO + IDENT_GEN
    # ============================================================

    grouped_sex_values = group_values_by_slug(
        values_by_column.get(sex_column, [])
    )

    for pair_column in pair_columns:
        grouped_pair_values = group_values_by_slug(
            values_by_column.get(pair_column, [])
        )

        for sex_key, raw_sex_values in grouped_sex_values.items():
            for pair_key, raw_pair_values in grouped_pair_values.items():
                alias = (
                    f"{sex_column}_{sex_key}"
                    f"__{pair_column}_{pair_key}"
                )

                select_parts.append(
                    f"""COUNT(*) FILTER (
                        WHERE {quote_ident(sex_column)} IN ({sql_in_list(raw_sex_values)})
                          AND {quote_ident(pair_column)} IN ({sql_in_list(raw_pair_values)})
                    ) AS {quote_ident(alias)}"""
                )

    query = f"""
        SELECT
            {",\n            ".join(select_parts)}
        FROM {quote_ident(schema)}.{quote_ident(table)}
        WHERE {quote_ident(date_column)} IS NOT NULL
          AND COALESCE(TRIM(LOWER({quote_ident(lesion_column)})), '') <> 'sim'
        GROUP BY
            DATE_TRUNC('month', {quote_ident(date_column)})::date,
            {quote_ident(municipality_column)},
            {quote_ident(state_column)}
        ORDER BY
            mes,
            {quote_ident(state_column)},
            {quote_ident(municipality_column)}
    """

    return query


def monthly_violence_counts_from_postgres(
    engine: Engine,
    schema: str,
    table: str,
    date_column: str = "DT_OCOR",
    municipality_column: str = "ID_MN_OCOR",
    state_column: str = "SG_UF_OCOR",
    lesion_column: str = "LES_AUTOP",
    sex_column: str = "CS_SEXO",
) -> pd.DataFrame:
    """
    Executa a agregação no Postgres e baixa apenas o resultado agregado
    para um DataFrame pandas.
    """

    _, _, categorical_columns = get_analysis_columns()

    values_by_column = get_distinct_values(
        engine=engine,
        schema=schema,
        table=table,
        columns=categorical_columns,
        lesion_column=lesion_column,
    )

    query = build_monthly_violence_query(
        schema=schema,
        table=table,
        values_by_column=values_by_column,
        date_column=date_column,
        municipality_column=municipality_column,
        state_column=state_column,
        lesion_column=lesion_column,
        sex_column=sex_column,
    )

    return pd.read_sql(text(query), con=engine)


def create_monthly_violence_counts_table(
    engine: Engine,
    source_schema: str,
    source_table: str,
    target_schema: str,
    target_table: str,
    date_column: str = "DT_OCOR",
    municipality_column: str = "ID_MN_OCOR",
    state_column: str = "SG_UF_OCOR",
    lesion_column: str = "LES_AUTOP",
    sex_column: str = "CS_SEXO",
    drop_if_exists: bool = True,
) -> None:
    """
    Cria uma tabela agregada diretamente no Postgres,
    sem baixar o resultado para pandas.
    """

    _, _, categorical_columns = get_analysis_columns()

    values_by_column = get_distinct_values(
        engine=engine,
        schema=source_schema,
        table=source_table,
        columns=categorical_columns,
        lesion_column=lesion_column,
    )

    select_query = build_monthly_violence_query(
        schema=source_schema,
        table=source_table,
        values_by_column=values_by_column,
        date_column=date_column,
        municipality_column=municipality_column,
        state_column=state_column,
        lesion_column=lesion_column,
        sex_column=sex_column,
    )

    statements = []

    if drop_if_exists:
        statements.append(
            f"DROP TABLE IF EXISTS {quote_ident(target_schema)}.{quote_ident(target_table)}"
        )

    statements.append(
        f"""
        CREATE TABLE {quote_ident(target_schema)}.{quote_ident(target_table)} AS
        {select_query}
        """
    )

    final_query = ";\n".join(statements) + ";"

    with engine.begin() as conn:
        conn.execute(text(final_query))


# ============================================================
# USO
# ============================================================

create_monthly_violence_counts_table(
    engine=engine,
    source_schema="sinan_viol",
    source_table="ocorrencias",
    target_schema="sinan_viol",
    target_table="contagens_mensais_violencia",
)


# ============================================================
# Conferir depois
# ============================================================

from sql_utils import print_table_head

target_schema_name = "sinan_viol"
target_table_name = "contagens_mensais_violencia"

print_table_head(
    engine,
    target_schema_name,
    target_table_name,
    rows=10,
)

       mes  ID_MN_OCOR SG_UF_OCOR  Total  CS_SEXO_f  CS_SEXO_i  CS_SEXO_m  LOCAL_OCOR_vazio  LOCAL_OCOR_bar_ou_similar  LOCAL_OCOR_comercio_servicos  LOCAL_OCOR_escola  LOCAL_OCOR_habitacao_coletiva  LOCAL_OCOR_ignorado  LOCAL_OCOR_industrias_construcao  LOCAL_OCOR_local_de_pratica_esportiva  LOCAL_OCOR_outro  LOCAL_OCOR_residencia  LOCAL_OCOR_via_publica  VIOL_FISIC_vazio  VIOL_FISIC_ignorado  VIOL_FISIC_nao  VIOL_FISIC_sim  VIOL_PSICO_vazio  VIOL_PSICO_ignorado  VIOL_PSICO_nao  VIOL_PSICO_sim  VIOL_TORT_vazio  VIOL_TORT_ignorado  VIOL_TORT_nao  VIOL_TORT_sim  VIOL_SEXU_vazio  VIOL_SEXU_ignorado  VIOL_SEXU_nao  VIOL_SEXU_sim  VIOL_TRAF_vazio  VIOL_TRAF_ignorado  VIOL_TRAF_nao  VIOL_TRAF_sim  VIOL_FINAN_vazio  VIOL_FINAN_ignorado  VIOL_FINAN_nao  VIOL_FINAN_sim  VIOL_NEGLI_vazio  VIOL_NEGLI_ignorado  VIOL_NEGLI_nao  VIOL_NEGLI_sim  VIOL_INFAN_vazio  VIOL_INFAN_ignorado  VIOL_INFAN_nao  VIOL_INFAN_sim  VIOL_LEGAL_vazio  VIOL_LEGAL_ignorado  VIOL_LEGAL_nao  VIOL_LEGAL_sim  VIOL_OUTR_vazi

In [113]:
def get_table_columns(
    engine: Engine,
    schema: str,
    table: str,
) -> list[str]:
    query = text("""
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema = :schema
          AND table_name = :table
        ORDER BY ordinal_position;
    """)

    with engine.connect() as conn:
        return [
            row.column_name
            for row in conn.execute(
                query,
                {"schema": schema, "table": table},
            )
        ]


def add_state_and_federal_rows_to_monthly_counts(
    engine: Engine,
    schema: str,
    table: str,
    date_column: str = "mes",
    municipality_column: str = "ID_MN_OCOR",
    state_column: str = "SG_UF_OCOR",
    federal_state_value: str = "0",
    federal_municipality_value: int = -999999,
    delete_existing_aggregates: bool = True,
) -> None:
    """
    Adiciona linhas estaduais e federais à tabela de contagens mensais.

    Convenção:
    - Linhas municipais:
        ID_MN_OCOR > 0
        SG_UF_OCOR = código positivo da UF em texto, ex: '41', '35', '50'

    - Linhas estaduais:
        ID_MN_OCOR = -SG_UF_OCOR
        SG_UF_OCOR = código positivo da UF em texto

    - Linha federal:
        ID_MN_OCOR = -999999
        SG_UF_OCOR = '1'

    Observação:
    Linhas com SG_UF_OCOR vazio, nulo ou não numérico são ignoradas
    na agregação estadual/federal.
    """

    all_columns = get_table_columns(
        engine=engine,
        schema=schema,
        table=table,
    )

    id_columns = {
        date_column,
        municipality_column,
        state_column,
    }

    count_columns = [
        column for column in all_columns
        if column not in id_columns
    ]

    if not count_columns:
        raise ValueError("Nenhuma coluna de contagem foi encontrada.")

    full_table_name = f"{quote_ident(schema)}.{quote_ident(table)}"

    insert_columns = [
        date_column,
        municipality_column,
        state_column,
        *count_columns,
    ]

    insert_columns_sql = ", ".join(
        quote_ident(column)
        for column in insert_columns
    )

    summed_count_columns_sql = ",\n            ".join(
        f"SUM({quote_ident(column)}) AS {quote_ident(column)}"
        for column in count_columns
    )

    valid_municipal_filter = f"""
        {quote_ident(municipality_column)} > 0
        AND {quote_ident(state_column)} IS NOT NULL
        AND TRIM({quote_ident(state_column)}) <> ''
        AND TRIM({quote_ident(state_column)}) <> :federal_state_value
        AND TRIM({quote_ident(state_column)}) ~ '^[0-9]+$'
    """

    delete_sql = ""

    if delete_existing_aggregates:
        delete_sql = f"""
            DELETE FROM {full_table_name}
            WHERE {quote_ident(municipality_column)} < 0
               OR TRIM({quote_ident(state_column)}) = :federal_state_value;
        """

    state_insert_sql = f"""
        INSERT INTO {full_table_name} (
            {insert_columns_sql}
        )
        SELECT
            {quote_ident(date_column)} AS {quote_ident(date_column)},
            -CAST(TRIM({quote_ident(state_column)}) AS BIGINT) AS {quote_ident(municipality_column)},
            TRIM({quote_ident(state_column)}) AS {quote_ident(state_column)},
            {summed_count_columns_sql}
        FROM {full_table_name}
        WHERE {valid_municipal_filter}
        GROUP BY
            {quote_ident(date_column)},
            TRIM({quote_ident(state_column)});
    """

    federal_insert_sql = f"""
        INSERT INTO {full_table_name} (
            {insert_columns_sql}
        )
        SELECT
            {quote_ident(date_column)} AS {quote_ident(date_column)},
            :federal_municipality_value AS {quote_ident(municipality_column)},
            :federal_state_value AS {quote_ident(state_column)},
            {summed_count_columns_sql}
        FROM {full_table_name}
        WHERE {valid_municipal_filter}
        GROUP BY
            {quote_ident(date_column)};
    """

    with engine.begin() as conn:
        if delete_existing_aggregates:
            conn.execute(
                text(delete_sql),
                {
                    "federal_state_value": federal_state_value,
                },
            )

        conn.execute(
            text(state_insert_sql),
            {
                "federal_state_value": federal_state_value,
            },
        )

        conn.execute(
            text(federal_insert_sql),
            {
                "federal_state_value": federal_state_value,
                "federal_municipality_value": federal_municipality_value,
            },
        )

In [114]:
add_state_and_federal_rows_to_monthly_counts(
    engine=engine,
    schema="sinan_viol",
    table="contagens_mensais_violencia",
)

In [115]:
# Outra que mandei o Codex fazer:

from sqlalchemy import text
from sqlalchemy.engine import Engine


def quote_ident(identifier: str) -> str:
    return '"' + identifier.replace('"', '""') + '"'


def create_violence_rates_by_population_table(
    engine: Engine,
    violence_schema: str,
    violence_table: str,
    pib_schema: str,
    pib_table: str,
    target_schema: str,
    target_table: str,
    violence_date_column: str = "mes",
    violence_municipality_column: str = "ID_MN_OCOR",
    violence_state_column: str = "SG_UF_OCOR",
    pib_municipality_column: str = "ID_MN_OCOR",
    pib_year_column: str = "ano",
    population_column: str = "populacao_proj_por_1000",
    drop_if_exists: bool = True,
) -> None:
    """
    Cria uma tabela nova comparando a tabela agregada de violência mensal
    com a tabela Pib_Geral.

    Para cada coluna de contagem da tabela de violência, cria uma coluna:

        coluna_por_1000hab = coluna / populacao_proj_por_1000

    O join é feito por:
        município
        ano extraído da coluna mes

    IMPORTANTE:
    Usa INNER JOIN. Portanto, se não existir par ano-município
    na Pib_Geral, a linha da tabela de contagem é descartada.
    """

    violence_columns = get_table_columns(
        engine=engine,
        schema=violence_schema,
        table=violence_table,
    )

    id_columns = {
        violence_date_column,
        violence_municipality_column,
        violence_state_column,
    }

    count_columns = [
        column for column in violence_columns
        if column not in id_columns
    ]

    if not count_columns:
        raise ValueError("Nenhuma coluna de contagem foi encontrada.")

    v = "v"
    p = "p"

    select_parts = [
        f"{v}.{quote_ident(violence_date_column)} AS {quote_ident(violence_date_column)}",
        f"EXTRACT(YEAR FROM {v}.{quote_ident(violence_date_column)})::int AS ano",
        f"{v}.{quote_ident(violence_municipality_column)} AS {quote_ident(violence_municipality_column)}",
        f"{v}.{quote_ident(violence_state_column)} AS {quote_ident(violence_state_column)}",
        f"{p}.{quote_ident(population_column)} AS {quote_ident(population_column)}",
    ]

    for column in count_columns:
        alias = f"{column}"

        select_parts.append(
            f"""
            CASE
                WHEN {p}.{quote_ident(population_column)} = 0 THEN NULL
                ELSE {v}.{quote_ident(column)}::numeric / {p}.{quote_ident(population_column)}::numeric
            END AS {quote_ident(alias)}
            """
        )

    drop_statement = ""

    if drop_if_exists:
        drop_statement = f"""
            DROP TABLE IF EXISTS {quote_ident(target_schema)}.{quote_ident(target_table)};
        """

    create_query = f"""
        {drop_statement}

        CREATE TABLE {quote_ident(target_schema)}.{quote_ident(target_table)} AS
        SELECT
            {", ".join(select_parts)}
        FROM {quote_ident(violence_schema)}.{quote_ident(violence_table)} AS {v}
        INNER JOIN {quote_ident(pib_schema)}.{quote_ident(pib_table)} AS {p}
            ON {v}.{quote_ident(violence_municipality_column)} = {p}.{quote_ident(pib_municipality_column)}
           AND EXTRACT(YEAR FROM {v}.{quote_ident(violence_date_column)})::int = {p}.{quote_ident(pib_year_column)}::int
        WHERE {p}.{quote_ident(population_column)} IS NOT NULL
        ORDER BY
            {v}.{quote_ident(violence_date_column)},
            {v}.{quote_ident(violence_state_column)},
            {v}.{quote_ident(violence_municipality_column)};
    """

    with engine.begin() as conn:
        conn.execute(text(create_query))

In [116]:
def cast_columns_to_double(
    engine: Engine,
    schema: str,
    table: str,
    columns: list[str],
) -> None:
    """
    Altera o tipo das colunas informadas para DOUBLE PRECISION.

    Exemplo:
        INTEGER -> DOUBLE PRECISION
        BIGINT  -> DOUBLE PRECISION
        NUMERIC -> DOUBLE PRECISION
        TEXT numérico -> DOUBLE PRECISION, se for conversível
    """

    if not columns:
        raise ValueError("A lista de colunas está vazia.")

    alter_parts = []

    for column in columns:
        alter_parts.append(
            f"""
            ALTER COLUMN {quote_ident(column)}
            TYPE DOUBLE PRECISION
            USING {quote_ident(column)}::DOUBLE PRECISION
            """
        )

    query = f"""
        ALTER TABLE {quote_ident(schema)}.{quote_ident(table)}
        {", ".join(alter_parts)};
    """

    with engine.begin() as conn:
        conn.execute(text(query))

In [117]:
def get_table_columns(
    engine: Engine,
    schema: str,
    table: str,
) -> list[str]:
    query = text("""
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema = :schema
          AND table_name = :table
        ORDER BY ordinal_position;
    """)

    with engine.connect() as conn:
        return [
            row.column_name
            for row in conn.execute(
                query,
                {"schema": schema, "table": table},
            )
        ]


def cast_count_columns_to_double(
    engine: Engine,
    schema: str,
    table: str,
    id_columns: set[str] | None = None,
) -> None:
    """
    Converte para DOUBLE PRECISION todas as colunas que não são identificadores.

    Útil para tabelas agregadas como:
        mes
        ID_MN_OCOR
        SG_UF_OCOR
        Total
        CS_SEXO_f
        ...
    """

    if id_columns is None:
        id_columns = {
            "mes",
            "ano",
            "ID_MN_OCOR",
            "SG_UF_OCOR",
            "ID_MUNICIP",
            "ID_MUNICIPIO",
        }

    columns = get_table_columns(
        engine=engine,
        schema=schema,
        table=table,
    )

    count_columns = [
        column for column in columns
        if column not in id_columns
    ]

    cast_columns_to_double(
        engine=engine,
        schema=schema,
        table=table,
        columns=count_columns,
    )

In [118]:
cast_count_columns_to_double(
    engine=engine,
    schema="sinan_viol",
    table="contagens_mensais_violencia",
)

In [119]:
create_violence_rates_by_population_table(
    engine=engine,
    violence_schema="sinan_viol",
    violence_table="contagens_mensais_violencia",
    pib_schema="norm_por_1k",
    pib_table="Pib_Geral",
    target_schema="sinan_viol",
    target_table="contagens_mensais_violencia_por_1000hab",
    violence_date_column="mes",
    violence_municipality_column="ID_MN_OCOR",
    violence_state_column="SG_UF_OCOR",
    pib_municipality_column="Cod",
    pib_year_column="Ano",
    population_column="populacao_proj_por_1000",
)

In [120]:
from sql_utils import print_table_head

target_schema_name = "sinan_viol"
target_table_name = "contagens_mensais_violencia_por_1000hab"

print_table_head(
    engine,
    target_schema_name,
    target_table_name,
    rows=10,
)

       mes  ano  ID_MN_OCOR SG_UF_OCOR  populacao_proj_por_1000    Total  CS_SEXO_f  CS_SEXO_i  CS_SEXO_m  LOCAL_OCOR_vazio  LOCAL_OCOR_bar_ou_similar  LOCAL_OCOR_comercio_servicos  LOCAL_OCOR_escola  LOCAL_OCOR_habitacao_coletiva  LOCAL_OCOR_ignorado  LOCAL_OCOR_industrias_construcao  LOCAL_OCOR_local_de_pratica_esportiva  LOCAL_OCOR_outro  LOCAL_OCOR_residencia  LOCAL_OCOR_via_publica  VIOL_FISIC_vazio  VIOL_FISIC_ignorado  VIOL_FISIC_nao  VIOL_FISIC_sim  VIOL_PSICO_vazio  VIOL_PSICO_ignorado  VIOL_PSICO_nao  VIOL_PSICO_sim  VIOL_TORT_vazio  VIOL_TORT_ignorado  VIOL_TORT_nao  VIOL_TORT_sim  VIOL_SEXU_vazio  VIOL_SEXU_ignorado  VIOL_SEXU_nao  VIOL_SEXU_sim  VIOL_TRAF_vazio  VIOL_TRAF_ignorado  VIOL_TRAF_nao  VIOL_TRAF_sim  VIOL_FINAN_vazio  VIOL_FINAN_ignorado  VIOL_FINAN_nao  VIOL_FINAN_sim  VIOL_NEGLI_vazio  VIOL_NEGLI_ignorado  VIOL_NEGLI_nao  VIOL_NEGLI_sim  VIOL_INFAN_vazio  VIOL_INFAN_ignorado  VIOL_INFAN_nao  VIOL_INFAN_sim  VIOL_LEGAL_vazio  VIOL_LEGAL_ignorado  VIOL_LEGAL_nao

chat, iutro problema, tenho uma tabela com essas e outras colunas DT_OCOR (TIMESTAMP), ID_MN_OCOR (BIGINT), CS_SEXO (TEXT), SG_UF_OCOR (TEXT), LOCAL_OCOR(TEXT), LES_AUTOP, VIOL_FISIC: TEXT, VIOL_PSICO: TEXT, VIOL_TORT: TEXT ,  VIOL_SEXU: TEXT, VIOL_TRAF: TEXT, VIOL_FINAN: TEXT, VIOL_NEGLI: TEXT, VIOL_INFAN: TEXT, VIOL_LEGAL: TEXT, VIOL_OUTR: TEXT, SEX_ASSEDI: TEXT, SEX_ESTUPR: TEXT, SEX_PUDOR: TEXT, SEX_PORNO: TEXT, SEX_EXPLO: TEXT, SEX_OUTRO: TEXT, SEX_ESPEC: TEXT, AUTOR_SEXO: TEXT, ORIENT_SEX: TEXT, IDENT_GEN: TEXT, VIOL_MOTIV: TEXT.

Eu quero fazer contagens mensais (DT_OCOR), por ID_MN_OCOR e SG_UF_OCOR. Descartando as colunas em que LES_AUTOP = Sim

e quero colunas que sejam o Total,  entradas chaves unitarias de cada uma das outras colunas no formato nome_da_coluna_chave, mais colunas que contam as combinação de chaves de:  CS_SEXO + LOCAL_OCOR, CS_SEXO + AUTOR_SEXO, CS_SEXO + VIOL*, CS_SEXO + SEX*, CS_SEXO + ORIENT_SEX, CS_SEXO + IDENT_GEN

SEX_ASSEDI: TEXT, SEX_ESTUPR: TEXT, SEX_PUDOR: TEXT, SEX_PORNO: TEXT, SEX_EXPLO: TEXT, SEX_OUTRO: TEXT, SEX_ESPEC: TEXT

## Limpando de municipios quie não precisamos